In [ ]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

## **Step 1: Parse the PDF Files**

In [2]:
from langchain_community.document_loaders import PyPDFLoader

def process_invoice(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    
    # Combine content from pages
    full_text = "\n".join([d.page_content for d in docs])
    
    # Invoke structured extraction
    return full_text

In [3]:
process_invoice("./invoices/demo-invoice-20tax-2.pdf")

Ignoring wrong pointing object 2 65536 (offset 0)


'Contoso Asia  \n123 Xia Street \nBeijing  \n100027 \nPhone: (0123) 4567 8901 \nVAT No: CN213 4444 66 \nOrg nr: 6005218080 \n \nINVOICE \n \nInvoice Address \nACME Inc \n44 Shore St \nMacduff \nAB4 1TX \n  \n \n \n \n \nINVOICE NUMBER 347003 \nINVOICE DATE 3/29/2025 \nDUE DATE 4/28/2025 \n  \n \n \nSHIPPING METHOD SHIPPING TERMS DELIVERY DATE \nDHL Free shipping. 3/29/2025 \n \nLINE ITEM # DESCRIPTION QTY UNIT \nPRICE LINE TOTAL \n  Your PO      000006    \n1 12345 SpeakerCable Speaker ca ble 10 650 300.00 195,000.00 \n2 55551 SurroundSoundReceive 450 320.00 144,000.00 \n3 12333 TelevisionM12037 Television M120 37 Silver 625 160.00 100,000.00 \n4 22333 Soundbar 350 120.00 42,000.00 \n5 17111 0.1 Cable 1 59.99 59.99 \n      \n      \n      \n      \n      \n  \nSubtotal   481,059.99 \nVAT rate 20.0% \nTotal VAT   96,212.00 \n Total  577,271.99 \nContoso Asia  \n123 Xia Street \nBeijing  \n100027\n \nPhone: (0123) 4567 8901 \nFax: (0123) 4567 8902 \ninfo@contosoasia.com'

## **Step 2: Define the Extraction Schema**

Using `pydantic` allows the LLM to enforce the structure, eliminating the need for `parse_json()` helper functions.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class InvoiceData(BaseModel):
    client_name: Optional[str] = Field(description="Client name")
    invoice_amount: Optional[float] = Field(description="Invoice amount")
    product_name: Optional[str] = Field(description="Product name")

# Bind to your model (e.g., using ChatGoogleGenerativeAI)
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
structured_llm = llm.with_structured_output(InvoiceData)